# Benchmark: 6D Reconstruction (Optimized - No Warmup)
This notebook benchmarks the execution speed of the 6D reconstruction workflow using only the **Current Optimized (Tensorised) Version** of the Cloud-in-Cell (CIC) implementation **without any warmup**.

This allows us to measure the exact startup and context initialization overhead for this version.

In [ ]:
# Device selection: change this to 'cpu', 'cuda', or 'mps'
device = "cpu"
max_epochs = 20

In [ ]:
import sys
import os
import time
import re
import glob
import platform
import subprocess
import pandas as pd
import numpy as np
import torch
import lightning as L

# Set repository root relative to dev/
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.append(repo_root)

print("Imports completed successfully.")

In [ ]:
def get_cpu_info():
    try:
        if platform.system() == "Windows":
            return platform.processor()
        elif platform.system() == "Darwin":
            return subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"]).decode().strip()
        elif platform.system() == "Linux":
            with open("/proc/cpuinfo", "r") as f:
                for line in f:
                    if "model name" in line:
                        return re.sub(".*model name.*:", "", line, 1).strip()
    except Exception:
        pass
    return platform.processor() or "Unknown CPU"

def get_ram_info():
    try:
        if platform.system() == "Darwin":
            mem_bytes = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"]).decode().strip())
            return f"{mem_bytes / (1024**3):.1f} GB"
        elif platform.system() == "Linux":
            with open("/proc/meminfo", "r") as f:
                for line in f:
                    if "MemTotal" in line:
                        mem_kb = int(line.split()[1])
                        return f"{mem_kb / (1024**2):.1f} GB"
    except Exception:
        pass
    return "Unknown RAM"

def get_gpu_info():
    gpus = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            gpus.append(torch.cuda.get_device_name(i))
    if torch.backends.mps.is_available():
        gpus.append("Apple Silicon Integrated GPU (MPS)")
    return ", ".join(gpus) if gpus else "None"

cpu_info = get_cpu_info()
ram_info = get_ram_info()
gpu_info = get_gpu_info()
os_info = f"{platform.system()} {platform.release()}"

system_label = f"{os_info} | {cpu_info} | RAM: {ram_info}"
if gpu_info != "None":
    system_label += f" | GPU: {gpu_info}"

print("System Information:")
print(f"OS:  {os_info}")
print(f"CPU: {cpu_info}")
print(f"RAM: {ram_info}")
print(f"GPU: {gpu_info}")
print(f"Label: {system_label}")

In [ ]:
from gpsr.datasets import split_dataset
from gpsr.modeling import GPSR6DLattice, GPSR
from gpsr.train import LitGPSR
from gpsr.beams import NNParticleBeamGenerator

# Path to the synthetic 6D dataset.
dset_path = os.path.expanduser("~/Documents/DESY/gpsr/docs/examples/example_data/example_datasets/reconstruction_6D.dset")
print(f"Loading dataset from {dset_path}...")
dset = torch.load(dset_path, weights_only=False)

train_k_ids = np.arange(0, len(dset.six_d_parameters), 2)
train_dset, test_dset = split_dataset(dset, train_k_ids)
train_loader = torch.utils.data.DataLoader(train_dset, batch_size=10)

p0c = 43.36e6  # reference momentum in eV/c
screens = train_dset.screens
l_quad = 0.11
l_tdc = 0.01
f_tdc = 1.3e9
phi_tdc = 0.0
l_bend = 0.3018
theta_on = -20.0 * 3.14 / 180.0
l1 = 0.790702
l2 = 0.631698
l3 = 0.889

gpsr_lattice = GPSR6DLattice(
    l_quad, l_tdc, f_tdc, phi_tdc, l_bend, theta_on, l1, l2, l3, *screens
)
print("Dataset loaded and GPSR6DLattice initialized.")

In [ ]:
# Monkeypatch set_lattice_parameters to clear cheetah's cache, avoiding shape mismatch errors
def set_lattice_parameters_with_cache_clearing(self, x: torch.Tensor) -> None:
    for elem in self.segment.elements:
        if hasattr(elem, "_cache"):
            elem._cache.clear()
        if hasattr(elem, "elements"):
            for sub_elem in elem.elements:
                if hasattr(sub_elem, "_cache"):
                    sub_elem._cache.clear()
    self.segment.SCAN_QUAD.k1.data = x[..., 0]
    self.segment.SCAN_TDC.voltage.data = x[..., 1]
    G = x[..., 2]
    bend_angle = torch.arcsin(self.l_bend * G)
    arc_length = bend_angle / G
    self.segment.SCAN_DIPOLE.angle.data = bend_angle
    self.segment.SCAN_DIPOLE.length.data = arc_length
    self.segment.SCAN_DIPOLE.dipole_e2.data = bend_angle
    self.segment.DIPOLE_TO_SCREEN.length.data = (
        self.l3 - self.l_bend / 2 / torch.cos(bend_angle)
    )

GPSR6DLattice.set_lattice_parameters = set_lattice_parameters_with_cache_clearing
print("Cache-clearing parameter update hook registered!")

In [ ]:
print(f"--- RUN: Current Optimized Version on {device.upper()} (No Warmup) ---")
torch.manual_seed(42)
np.random.seed(42)
L.seed_everything(42)

for elem in gpsr_lattice.segment.elements:
    if hasattr(elem, "_cache"):
        elem._cache.clear()
    if hasattr(elem, "elements"):
        for sub_elem in elem.elements:
            if hasattr(sub_elem, "_cache"):
                sub_elem._cache.clear()

gpsr_model_opt = GPSR(NNParticleBeamGenerator(10000, p0c), gpsr_lattice)
litgpsr_opt = LitGPSR(gpsr_model_opt)

accelerator = "gpu" if device in ["cuda", "mps"] else "cpu"
devices = [0] if device in ["cuda", "mps"] else 1

trainer_opt = L.Trainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    enable_checkpointing=False,
    logger=False
)

t0 = time.perf_counter()
trainer_opt.fit(model=litgpsr_opt, train_dataloaders=train_loader)
t_opt = time.perf_counter() - t0
print(f"\nCurrent Optimized execution time (No Warmup): {t_opt:.4f} seconds")

In [ ]:
# Save results
run_data = [{
    "System": system_label,
    "Device": device.upper(),
    "Current Optimized (s)": t_opt
}]
df_run = pd.DataFrame(run_data)

system_slug = re.sub(r'[^a-z0-9]+', '_', system_label.lower()).strip('_')
results_dir = os.path.join(repo_root, "dev", "benchmark_results")
os.makedirs(results_dir, exist_ok=True)
csv_filename = os.path.join(results_dir, f"reconstruction_6d_results_opt_no_warmup_{device.lower()}_{system_slug}.csv")
df_run.to_csv(csv_filename, index=False)
print(f"Saved current run results to {csv_filename}")